# Example Selection for Mushroom Classification

## Model Selection and Approach

This notebook is used to generate **real, valid examples** of edible and poisonous mushrooms directly from the dataset.

Although multiple models were implemented in this project (Decision Tree, Random Forest, and Naive Bayes), we use the **Decision Tree model** for example generation and application deployment. Both Decision Tree and Random Forest achieved perfect performance on the dataset; however, the Decision Tree was selected because it is **simpler and more interpretable**, making it easier to understand how predictions are made based on input features.

Instead of manually guessing input values, we extract examples directly from the dataset and pass them through the trained Decision Tree model. This ensures that all examples used in our application are **consistent with the model’s predictions** and not randomly selected.

The goal of this notebook is to:
- Select real mushroom instances from the dataset  
- Pass them through the trained Decision Tree model  
- Identify examples that are correctly classified as edible or poisonous  
- Convert them into human-readable values for use in the Streamlit application  

This process guarantees that the examples used for testing and demonstration are accurate, reproducible, and aligned with the model’s behavior.

## Dataset and Model Setup

In this step, we load the original mushroom dataset along with the trained Decision Tree model and preprocessing components.

The dataset is sourced from the UCI Mushroom Dataset and contains categorical features describing physical characteristics of mushrooms, such as cap shape, odor, gill color, and habitat. Each mushroom is labeled as either edible (`e`) or poisonous (`p`).

Since the dataset does not include column headers by default, we manually assign the correct feature names to ensure consistency with the model training process.

We also load:
- The trained Decision Tree model (`dt`)
- The label encoders used during preprocessing

The label encoders are important because the model was trained on encoded (numerical) representations of categorical values. To generate valid predictions, we must apply the same transformations to any input data.

This setup ensures that all subsequent predictions are made using the same structure and encoding as the original training pipeline.

In [1]:
import pandas as pd
import pickle
import numpy as np

orig = pd.read_csv("../data/mushrooms.csv", header=None)

orig.columns = [
    "class", "cap-shape", "cap-surface", "cap-color", "bruises", "odor",
    "gill-attachment", "gill-spacing", "gill-size", "gill-color",
    "stalk-shape", "stalk-root", "stalk-surface-above-ring",
    "stalk-surface-below-ring", "stalk-color-above-ring",
    "stalk-color-below-ring", "veil-type", "veil-color",
    "ring-number", "ring-type", "spore-print-color",
    "population", "habitat"
]

label_encoders = pickle.load(open("../models/label_encoders.pkl", "rb"))
dt = pickle.load(open("../models/decision_tree.pkl", "rb"))

## Feature Ordering and Prediction Function

In this step, we define the exact feature order used by the trained model and create a function to generate predictions from raw dataset rows.

The `feature_order` list is critical because machine learning models expect input features in the **same order as during training**. Any mismatch in ordering or number of features can lead to incorrect predictions or errors. Notably, the `veil-type` feature is excluded because it does not provide useful information (it has the same value for all samples) and was not used during model training.

The `test_row` function performs the following steps:
1. Takes a single row from the dataset
2. Encodes each categorical feature using the corresponding label encoder
3. Arranges the encoded values in the correct feature order
4. Passes the encoded input into the trained Decision Tree model
5. Converts the predicted output back to its original label (`e` or `p`)

This ensures that predictions are made in a way that is fully consistent with the preprocessing and training pipeline, allowing us to accurately validate examples from the dataset.

In [2]:
feature_order = [
    "cap-shape", "cap-surface", "cap-color", "bruises", "odor",
    "gill-attachment", "gill-spacing", "gill-size", "gill-color",
    "stalk-shape", "stalk-root", "stalk-surface-above-ring",
    "stalk-surface-below-ring", "stalk-color-above-ring",
    "stalk-color-below-ring",
    "veil-color",
    "ring-number", "ring-type", "spore-print-color",
    "population", "habitat"
]

def test_row(row):
    encoded = [
        label_encoders[col].transform([row[col]])[0]
        for col in feature_order
    ]
    input_array = np.array(encoded).reshape(1, -1)
    pred = dt.predict(input_array)[0]
    return label_encoders["class"].inverse_transform([pred])[0]

## Extracting Edible Examples from the Dataset

In this step, we identify real examples of edible mushrooms by iterating through the dataset and applying our trained Decision Tree model to each row.

Instead of relying solely on the original labels in the dataset, we use the model’s prediction (`test_row`) to determine whether a mushroom is classified as edible. This ensures that the selected examples are not only labeled as edible, but are also **correctly predicted as edible by the model**.

We collect the first five such examples to create a small set of representative edible mushrooms. These examples are later used for testing and demonstrating the model within the Streamlit application.

This approach guarantees that the examples are:
- Drawn directly from real dataset instances  
- Consistent with the trained Decision Tree model’s behavior  
- Reliable for demonstration and evaluation purposes

In [3]:
edible_examples = []

for i in range(len(orig)):
    row = orig.iloc[i]
    if test_row(row) == "e":
        edible_examples.append(row)

    if len(edible_examples) == 5:
        break

pd.DataFrame(edible_examples)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g
5,e,x,y,y,t,a,f,c,b,n,...,s,w,w,p,w,o,p,k,n,g
6,e,b,s,w,t,a,f,c,b,g,...,s,w,w,p,w,o,p,k,n,m


## Mapping Encoded Values to Human-Readable Features

The mushroom dataset represents feature values using abbreviated codes (e.g., `x`, `s`, `k`), which are not intuitive for interpretation or user interaction.

To make the results understandable and usable in our Streamlit application, we define a mapping (`label_maps`) that converts these encoded values into meaningful, human-readable descriptions. For example:
- `x → convex` (cap shape)
- `k → black` (gill color)
- `p → pungent` (odor)

This mapping serves two main purposes:
1. **Interpretability**: It allows us to clearly understand the characteristics of each mushroom example without relying on encoded symbols.
2. **UI Consistency**: It ensures that the values displayed in the notebook match the dropdown options in the Streamlit app, which uses the same mappings.

By applying this mapping, we can convert raw dataset rows into readable feature descriptions that can be directly used for testing, demonstration, and analysis.

In [4]:
label_maps = {
    'class': {'e': 'edible', 'p': 'poisonous'},
    'cap-shape': {'b': 'bell', 'c': 'conical', 'f': 'flat', 'k': 'knobbed', 's': 'sunken', 'x': 'convex'},
    'cap-surface': {'f': 'fibrous', 'g': 'grooves', 's': 'smooth', 'y': 'scaly'},
    'cap-color': {'b': 'buff', 'c': 'cinnamon', 'e': 'red', 'g': 'gray', 'n': 'brown', 'p': 'pink', 'r': 'green', 'u': 'purple', 'w': 'white', 'y': 'yellow'},
    'bruises': {'f': 'no', 't': 'yes'},
    'odor': {'a': 'almond', 'c': 'creosote', 'f': 'foul', 'l': 'anise', 'm': 'musty', 'n': 'none', 'p': 'pungent', 's': 'spicy', 'y': 'fishy'},
    'gill-attachment': {'a': 'attached', 'f': 'free'},
    'gill-spacing': {'c': 'close', 'w': 'crowded'},
    'gill-size': {'b': 'broad', 'n': 'narrow'},
    'gill-color': {'b': 'buff', 'e': 'red', 'g': 'gray', 'h': 'chocolate', 'k': 'black', 'n': 'brown', 'o': 'orange', 'p': 'pink', 'r': 'green', 'u': 'purple', 'w': 'white', 'y': 'yellow'},
    'stalk-shape': {'e': 'enlarging', 't': 'tapering'},
    'stalk-root': {'b': 'bulbous', 'c': 'club', 'e': 'equal', 'r': 'rooted', '?': 'missing'},
    'stalk-surface-above-ring': {'f': 'fibrous', 'k': 'silky', 's': 'smooth', 'y': 'scaly'},
    'stalk-surface-below-ring': {'f': 'fibrous', 'k': 'silky', 's': 'smooth', 'y': 'scaly'},
    'stalk-color-above-ring': {'b': 'buff', 'c': 'cinnamon', 'e': 'red', 'g': 'gray', 'n': 'brown', 'o': 'orange', 'p': 'pink', 'w': 'white', 'y': 'yellow'},
    'stalk-color-below-ring': {'b': 'buff', 'c': 'cinnamon', 'e': 'red', 'g': 'gray', 'n': 'brown', 'o': 'orange', 'p': 'pink', 'w': 'white', 'y': 'yellow'},
    'veil-type': {'p': 'partial', 'u': 'universal'},
    'veil-color': {'n': 'brown', 'o': 'orange', 'w': 'white', 'y': 'yellow'},
    'ring-number': {'n': 'none', 'o': 'one', 't': 'two'},
    'ring-type': {'e': 'evanescent', 'f': 'flaring', 'l': 'large', 'n': 'none', 'p': 'pendant'},
    'spore-print-color': {'b': 'buff', 'h': 'chocolate', 'k': 'black', 'n': 'brown', 'o': 'orange', 'r': 'green', 'u': 'purple', 'w': 'white', 'y': 'yellow'},
    'population': {'a': 'abundant', 'c': 'clustered', 'n': 'numerous', 's': 'scattered', 'v': 'several', 'y': 'solitary'},
    'habitat': {'d': 'woods', 'g': 'grasses', 'l': 'leaves', 'm': 'meadows', 'p': 'paths', 'u': 'urban', 'w': 'waste'}
}

## Converting Edible Examples to Readable Format

After extracting edible examples from the dataset, we convert them into a human-readable format using the previously defined `label_maps`.

Each row originally contains encoded values (e.g., `x`, `k`, `n`), which are transformed into descriptive labels such as “convex,” “black,” and “brown.” This makes the examples easier to interpret and directly usable in the Streamlit application.

We store the decoded results in a DataFrame to present them in a structured table format. This allows us to:
- Clearly visualize the characteristics of each edible mushroom  
- Easily copy values for testing in the application  
- Use the table for documentation or presentation purposes  

These examples are not randomly generated; they are real data points from the dataset that have been validated through the Decision Tree model’s prediction pipeline.

In [5]:
def decode_row(row):
    decoded = {}
    for col in row.index:
        if col in label_maps:
            decoded[col] = label_maps[col].get(row[col], row[col])
    return decoded

In [6]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

In [7]:
# Convert edible examples to readable table
edible_df = pd.DataFrame([decode_row(row) for row in edible_examples])
edible_df = edible_df.drop(columns=["veil-type"], errors="ignore")

print("EDIBLE EXAMPLES:")
display(edible_df)

EDIBLE EXAMPLES:


,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,stalk-root,stalk-surface-above-ring,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,edible,convex,smooth,yellow,yes,almond,free,close,broad,black,enlarging,club,smooth,smooth,white,white,white,one,pendant,brown,numerous,grasses
1,edible,bell,smooth,white,yes,anise,free,close,broad,brown,enlarging,club,smooth,smooth,white,white,white,one,pendant,brown,numerous,meadows
2,edible,convex,smooth,gray,no,none,free,crowded,broad,black,tapering,equal,smooth,smooth,white,white,white,one,evanescent,brown,abundant,grasses
3,edible,convex,scaly,yellow,yes,almond,free,close,broad,brown,enlarging,club,smooth,smooth,white,white,white,one,pendant,black,numerous,grasses
4,edible,bell,smooth,white,yes,almond,free,close,broad,gray,enlarging,club,smooth,smooth,white,white,white,one,pendant,black,numerous,meadows


## Extracting Poisonous Examples from the Dataset

In this step, we identify examples of poisonous mushrooms by iterating through the dataset and applying the trained model to each row.

Similar to the edible examples, we do not rely solely on the dataset’s original labels. Instead, we use the model’s prediction function (`test_row`) to determine whether a mushroom is classified as poisonous. This ensures that the selected examples are **consistent with the Decision Tree model’s behavior**.

We collect the first five examples that the model predicts as poisonous. These examples are important for:
- Demonstrating how the Decision Tree model identifies dangerous mushrooms  
- Testing the application with realistic inputs  
- Comparing patterns between edible and poisonous mushrooms  

By selecting examples in this way, we ensure that all poisonous cases used in our project are grounded in real data and validated through the trained Decision Tree model, rather than being manually chosen or guessed.

In [8]:
poisonous_examples = []

for i in range(len(orig)):
    row = orig.iloc[i]
    if test_row(row) == "p":
        poisonous_examples.append(row)

    if len(poisonous_examples) == 5:
        break

pd.DataFrame(poisonous_examples)

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,stalk-root,stalk-surface-above-ring,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,e,e,s,s,w,w,p,w,o,p,k,s,u
3,p,x,y,w,t,p,f,c,n,n,e,e,s,s,w,w,p,w,o,p,k,s,u
8,p,x,y,w,t,p,f,c,n,p,e,e,s,s,w,w,p,w,o,p,k,v,g
13,p,x,y,w,t,p,f,c,n,k,e,e,s,s,w,w,p,w,o,p,n,v,u
17,p,x,s,n,t,p,f,c,n,n,e,e,s,s,w,w,p,w,o,p,k,s,g


## Converting Poisonous Examples to Readable Format

After extracting poisonous examples, we convert them into a human-readable format using the `label_maps`.

The original dataset uses encoded values (such as `n`, `p`, `k`), which are translated into descriptive labels like “brown,” “pungent,” and “black.” This makes the examples easier to interpret and aligns them with the dropdown options used in the Streamlit application.

The decoded examples are stored in a DataFrame and displayed as a table. This allows us to:
- Clearly examine the characteristics of poisonous mushrooms  
- Compare them with edible examples  
- Use them directly for testing and demonstration in the application  

These examples are derived directly from the dataset and verified using the trained Decision Tree model, ensuring that they are accurate and consistent with the Decision Tree model’s predictions.

In [9]:
# Convert poisonous examples to readable table
poisonous_df = pd.DataFrame([decode_row(row) for row in poisonous_examples])
poisonous_df = poisonous_df.drop(columns=["veil-type"], errors="ignore")

print("POISONOUS EXAMPLES:")
display(poisonous_df)

POISONOUS EXAMPLES:


,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,stalk-root,stalk-surface-above-ring,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,poisonous,convex,smooth,brown,yes,pungent,free,close,narrow,black,enlarging,equal,smooth,smooth,white,white,white,one,pendant,black,scattered,urban
1,poisonous,convex,scaly,white,yes,pungent,free,close,narrow,brown,enlarging,equal,smooth,smooth,white,white,white,one,pendant,black,scattered,urban
2,poisonous,convex,scaly,white,yes,pungent,free,close,narrow,pink,enlarging,equal,smooth,smooth,white,white,white,one,pendant,black,several,grasses
3,poisonous,convex,scaly,white,yes,pungent,free,close,narrow,black,enlarging,equal,smooth,smooth,white,white,white,one,pendant,brown,several,urban
4,poisonous,convex,smooth,brown,yes,pungent,free,close,narrow,brown,enlarging,equal,smooth,smooth,white,white,white,one,pendant,black,scattered,grasses
